In [ ]:
# ── Bootstrap: Download + Extract OmniGeoFusion from S3 ──
import os
from google.colab import userdata

# AWS credentials from Colab secrets
AWS_KEY    = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET = userdata.get('AWS_SECRET_ACCESS_KEY')

os.environ['AWS_ACCESS_KEY_ID']     = AWS_KEY
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'

BUCKET   = 'omnigeofusion-data-288528696055'
REPO_DIR = '/content/omnigeofusion'

# Install awscli
os.system('pip install awscli -q')

# Download bundle from S3
print('Downloading repo from S3...')
os.system(f'''aws s3 cp \
    s3://{BUCKET}/repo/omnigeofusion.tar.gz \
    /tmp/omnigeofusion.tar.gz''')

# Extract
os.makedirs(REPO_DIR, exist_ok=True)
os.system('tar -xzf /tmp/omnigeofusion.tar.gz -C /content/')

# Verify
print(f'\nProject structure:')
for item in sorted(os.listdir(REPO_DIR)):
    print(f'  {item}/')

import sys
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print(f'\n✅ Project ready at {REPO_DIR}')
print(f'Working dir: {os.getcwd()}')
print(f'\nNotebooks available:')
for nb in os.listdir(f'{REPO_DIR}/notebooks'):
    print(f'  notebooks/{nb}')


Project structure:
  README.md/
  configs/
  docker/
  notebooks/
  requirements.txt/
  src/
  tests/

✅ Project ready at /content/omnigeofusion
Working dir: /content/omnigeofusion

Notebooks available:
  notebooks/00_prepare_data.ipynb
  notebooks/01_ssl_pretrain.ipynb
  notebooks/02_finetune.ipynb


In [ ]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/gdrive')

import torch, os
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

os.makedirs('/gdrive/MyDrive/omnigeofusion/checkpoints/ssl', exist_ok=True)
os.makedirs('/gdrive/MyDrive/omnigeofusion/checkpoints/finetune', exist_ok=True)
print('✅ Drive mounted')

Mounted at /gdrive
GPU: Tesla T4
VRAM: 15.6GB
✅ Drive mounted


In [ ]:
# ── Install Dependencies ──
import subprocess, os

VENV_PYTHON = '/content/venv/bin/python3'
VENV_PIP    = '/content/venv/bin/pip'

os.system('apt-get install -y libgeos-dev libgdal-dev -q')
os.environ['MPLBACKEND'] = 'agg'

# Create venv
os.system('python3 -m venv /content/venv')
os.system('curl -sS https://bootstrap.pypa.io/get-pip.py | /content/venv/bin/python3')

def install(packages):
    r = subprocess.run(
        [VENV_PIP, 'install', '-q'] + packages,
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f'Error: {r.stderr[-300:]}')
    return r.returncode == 0

print('Step 1: terratorch...')
install(['terratorch'])

print('Step 2: torch (force reinstall)...')
install(['--force-reinstall', 'torch==2.2.0', 'torchvision==0.17.0'])

print('Step 3: dependencies...')
install([
    'transformers==4.40.0', 'timm', 'einops',
    'mlflow==2.14.3', 'boto3', 'rasterio',
    'scipy', 'pyyaml', 'huggingface_hub==0.20.3',
    'pandas', 'tqdm', 'pyproj',
])

print('Step 4: pin versions...')
install(['numpy==1.26.4', 'protobuf==3.20.3', 'setuptools==69.5.1'])

os.system('pip install -q boto3 rasterio')

# Verify
r = subprocess.run([VENV_PYTHON, '-c', '''
import os; os.environ["MPLBACKEND"] = "agg"
import numpy as np, torch
from terratorch.registry import BACKBONE_REGISTRY
print(f"numpy:      {np.__version__}")
print(f"torch:      {torch.__version__}")
print("terratorch: OK")
'''], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr[-200:])
print('✅ Venv ready')

Step 1: terratorch...
Step 2: torch (force reinstall)...
Step 3: dependencies...
Step 4: pin versions...
numpy:      1.26.4
torch:      2.2.0+cu121
terratorch: OK

✅ Venv ready


In [ ]:
# ── Verify Prithvi-EO-2.0-300M ──
import subprocess, os
from google.colab import userdata

VENV_PYTHON = '/content/venv/bin/python3'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')

r = subprocess.run([VENV_PYTHON, '-c', f'''
import os, torch
os.environ["MPLBACKEND"]            = "agg"
os.environ["AWS_ACCESS_KEY_ID"]     = "{AWS_KEY}"
os.environ["AWS_SECRET_ACCESS_KEY"] = "{AWS_SECRET}"
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

from terratorch.registry import BACKBONE_REGISTRY

print("Loading Prithvi-EO-2.0-300M...")
encoder = BACKBONE_REGISTRY.build(
    "prithvi_eo_v2_300",
    pretrained=True,
    num_frames=1,
    in_chans=6,
)
params = sum(p.numel() for p in encoder.parameters())
print(f"✅ Loaded: {{params/1e6:.0f}}M parameters")

x = torch.zeros(1, 6, 224, 224)
with torch.no_grad():
    out = encoder(x)

feat = out[-1] if isinstance(out, (list,tuple)) else out
emb  = feat[:,1:,:].mean(1) if feat.dim()==3 else feat.mean([2,3])
print(f"✅ Embedding: {{emb.shape}}")

if torch.cuda.is_available():
    encoder = encoder.cuda()
    with torch.no_grad():
        out_gpu = encoder(x.cuda())
    feat_gpu = out_gpu[-1] if isinstance(out_gpu,(list,tuple)) else out_gpu
    emb_gpu  = feat_gpu[:,1:,:].mean(1) if feat_gpu.dim()==3 else feat_gpu.mean([2,3])
    print(f"✅ GPU: {{emb_gpu.shape}} on {{torch.cuda.get_device_name(0)}}")

del encoder
torch.cuda.empty_cache()
print("✅ Prithvi ready")
'''], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr[-300:])

Loading Prithvi-EO-2.0-300M...
✅ Loaded: 304M parameters
✅ Embedding: torch.Size([1, 1024])
✅ GPU: torch.Size([1, 1024]) on Tesla T4
✅ Prithvi ready

STDERR: hvi_EO_V2_300M.pt:  96%|█████████▌| 1.27G/1.33G [00:21<00:01, 31.4MB/s]
Prithvi_EO_V2_300M.pt:  97%|█████████▋| 1.29G/1.33G [00:22<00:01, 34.5MB/s]
Prithvi_EO_V2_300M.pt:  99%|█████████▉| 1.31G/1.33G [00:22<00:00, 45.5MB/s]
Prithvi_EO_V2_300M.pt: 100%|██████████| 1.33G/1.33G [00:22<00:00, 59.2MB/s]



In [ ]:
# ── Download ALL Data to GDrive (once) + copy to /content ──
import boto3, json, os, random, shutil
from pathlib import Path
from tqdm import tqdm
from google.colab import userdata

AWS_KEY    = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET = userdata.get('AWS_SECRET_ACCESS_KEY')

s3 = boto3.client('s3',
    region_name='us-east-1',
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET
)

BUCKET      = 'omnigeofusion-data-288528696055'
GDRIVE_DIR  = Path('/gdrive/MyDrive/omnigeofusion/data')  # ← PERSISTENT
COLAB_DIR   = Path('/content/omnigeofusion_data')          # ← FAST ACCESS
MAX_PATCHES = None   # ← ALL patches, no limit
TRAIN_RATIO = 0.85
AREAS       = ['amsterdam', 'rotterdam', 'flevoland']
MODALITIES  = ['sentinel2', 'sentinel1', 'lidar', 'thermal']

GDRIVE_DIR.mkdir(parents=True, exist_ok=True)
COLAB_DIR.mkdir(parents=True, exist_ok=True)

print(f'GDrive: {GDRIVE_DIR}')
print(f'Colab:  {COLAB_DIR}')
print(f'Max patches: ALL (no limit)')

# ── Helper functions ──────────────────────────────────────────
def download_index(modality, area):
    key  = f'netherlands/{modality}/index/{area}_index.json'
    path = GDRIVE_DIR / 'indexes' / f'{area}_{modality}.json'
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        try:
            s3.download_file(BUCKET, key, str(path))
        except Exception as e:
            print(f'    ❌ {area}_{modality}: {e}')
            return []
    return json.loads(path.read_text())

def download_patch(s3_key, gdrive_path, colab_path):
    """Download to GDrive first (persistent), then copy to /content (fast)."""
    # Step 1: Download to GDrive if not already there
    if not gdrive_path.exists():
        try:
            gdrive_path.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file(BUCKET, s3_key, str(gdrive_path))
        except:
            return False
    # Step 2: Copy from GDrive to /content for fast training access
    if not colab_path.exists():
        colab_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(gdrive_path), str(colab_path))
    return True

# ── Step 1: Download raster indexes ───────────────────────────
print('\n=== Step 1: Raster indexes ===')
indexes = {}
for mod in MODALITIES:
    indexes[mod] = {}
    for area in AREAS:
        idx = download_index(mod, area)
        indexes[mod][area] = idx
        print(f'  {mod}/{area}: {len(idx)} patches')

# ── Step 2: OSM indexes ───────────────────────────────────────
print('\n=== Step 2: OSM indexes ===')
osm_indexes = {}
for area in AREAS:
    key  = f'netherlands/osm/index/{area}_osm.json'
    path = GDRIVE_DIR / 'indexes' / f'{area}_osm.json'
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        s3.download_file(BUCKET, key, str(path))
    data = json.loads(path.read_text())
    osm_indexes[area] = {d['patch_id']: d for d in data}
    print(f'  OSM/{area}: {len(data)} patches')

# ── Step 3: IoT indexes ───────────────────────────────────────
print('\n=== Step 3: IoT indexes ===')
iot_indexes = {}
for area in AREAS:
    key  = f'netherlands/iot/index/{area}_iot.json'
    path = GDRIVE_DIR / 'indexes' / f'{area}_iot.json'
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        try:
            s3.download_file(BUCKET, key, str(path))
        except:
            print(f'  IoT/{area}: not found — using zeros')
            iot_indexes[area] = []
            continue
    data = json.loads(path.read_text())
    iot_indexes[area] = data
    print(f'  IoT/{area}: {len(data)} readings')

# ── Step 4: Download ALL raster patches to GDrive ─────────────
print('\n=== Step 4: Download ALL patches to GDrive ===')
downloaded = {mod: {area: [] for area in AREAS} for mod in MODALITIES}

for mod in MODALITIES:
    for area in AREAS:
        # Use ALL patches — no limit
        idx     = indexes[mod][area]
        success = 0
        print(f'\n{mod}/{area}: {len(idx)} patches')
        for entry in tqdm(idx, ncols=60):
            gdrive_path = GDRIVE_DIR / entry['s3_key']
            colab_path  = COLAB_DIR  / entry['s3_key']
            if download_patch(entry['s3_key'], gdrive_path, colab_path):
                entry['local_path']  = str(colab_path)
                entry['gdrive_path'] = str(gdrive_path)
                downloaded[mod][area].append(entry)
                success += 1
        print(f'  ✅ {success}/{len(idx)} saved to GDrive')

# ── Step 5: Build multimodal training index ───────────────────
print('\n=== Step 5: Build training index ===')

MOD_KEYS = {
    'sentinel2': 's2_t1_path',
    'sentinel1': 'sar_path',
    'lidar':     'lidar_path',
    'thermal':   'thermal_path',
}

lookup = {area: {} for area in AREAS}

for mod in MODALITIES:
    for area in AREAS:
        for entry in downloaded[mod][area]:
            key = (entry['row'], entry['col'])
            if key not in lookup[area]:
                lookup[area][key] = {
                    'patch_id': entry['patch_id'],
                    'tile':     entry['tile'],
                    'area':     area,
                    'date':     entry['date'],
                    'row':      entry['row'],
                    'col':      entry['col'],
                    'day_gap':  90,
                }
            path_key = MOD_KEYS.get(mod)
            if path_key:
                lookup[area][key][path_key] = entry['local_path']

# Add OSM features per patch
for area in AREAS:
    for key, sample in lookup[area].items():
        pid = sample['patch_id']
        if pid in osm_indexes.get(area, {}):
            osm = osm_indexes[area][pid]
            sample['osm_features'] = [
                osm.get('building_count',       0),
                osm.get('building_density',     0),
                osm.get('mean_building_height', 0),
                osm.get('building_coverage',    0),
                osm.get('road_count',           0),
                osm.get('road_density',         0),
                osm.get('waterway_count',       0),
                osm.get('lu_residential',       0),
                osm.get('lu_farmland',          0),
                osm.get('lu_forest',            0),
                osm.get('lu_water',             0),
                osm.get('lu_commercial',        0),
                osm.get('lu_industrial',        0),
                osm.get('urban_score',          0),
                osm.get('population_proxy',     0),
                osm.get('total_building_area',  0),
            ]
        else:
            sample['osm_features'] = [0.0] * 16

# Add IoT features per patch (area-level aggregate)
for area in AREAS:
    iot_data   = iot_indexes.get(area, [])
    if iot_data:
        iot_entry  = iot_data[0]
        iot_vector = [
            iot_entry.get('NO2_mean',  0),
            iot_entry.get('NO2_min',   0),
            iot_entry.get('NO2_max',   0),
            iot_entry.get('PM10_mean', 0),
            iot_entry.get('PM10_min',  0),
            iot_entry.get('PM10_max',  0),
            iot_entry.get('O3_mean',   0),
            iot_entry.get('O3_min',    0),
            iot_entry.get('O3_max',    0),
            iot_entry.get('PM25_mean', 0),
            iot_entry.get('PM25_min',  0),
            iot_entry.get('PM25_max',  0),
            0, 0, 0, 0,  # water/weather placeholders
        ]
    else:
        iot_vector = [0.0] * 16

    for key, sample in lookup[area].items():
        sample['iot_features'] = iot_vector

# Flatten + train/val split
all_samples = []
for area in AREAS:
    for key, sample in lookup[area].items():
        if 's2_t1_path' in sample:
            all_samples.append(sample)

random.shuffle(all_samples)
split       = int(len(all_samples) * TRAIN_RATIO)
train_index = all_samples[:split]
val_index   = all_samples[split:]

# Save indexes to GDrive (persistent) + /content (fast)
for idx_data, name in [(train_index, 'train'), (val_index, 'val')]:
    (GDRIVE_DIR / f'{name}_index.json').write_text(
        json.dumps(idx_data, indent=2)
    )
    (COLAB_DIR / f'{name}_index.json').write_text(
        json.dumps(idx_data, indent=2)
    )
    print(f'✅ {name}_index.json saved to GDrive + /content')

# Summary
osm_count = sum(1 for s in train_index if any(v != 0 for v in s.get('osm_features', [0])))
iot_count = sum(1 for s in train_index if any(v != 0 for v in s.get('iot_features', [0])))

print(f'\n=== SUMMARY ===')
print(f'Total samples:  {len(all_samples)}')
print(f'Train:          {len(train_index)}')
print(f'Val:            {len(val_index)}')
print(f'OSM coverage:   {osm_count}/{len(train_index)} patches')
print(f'IoT coverage:   {iot_count}/{len(train_index)} patches')
print(f'\nGDrive size: {os.popen(f"du -sh {GDRIVE_DIR}").read().strip()}')
print('\n✅ ALL data saved to GDrive — never download again!')
print('   Next sessions: data loads from GDrive in ~2-3 mins')

GDrive: /gdrive/MyDrive/omnigeofusion/data
Colab:  /content/omnigeofusion_data
Max patches: ALL (no limit)

=== Step 1: Raster indexes ===
  sentinel2/amsterdam: 1666 patches
  sentinel2/rotterdam: 1666 patches
  sentinel2/flevoland: 1666 patches
  sentinel1/amsterdam: 1194 patches
  sentinel1/rotterdam: 814 patches
  sentinel1/flevoland: 1112 patches
  lidar/amsterdam: 1584 patches
  lidar/rotterdam: 289 patches
  lidar/flevoland: 981 patches
  thermal/amsterdam: 419 patches
  thermal/rotterdam: 1132 patches
  thermal/flevoland: 501 patches

=== Step 2: OSM indexes ===
  OSM/amsterdam: 1666 patches
  OSM/rotterdam: 1666 patches
  OSM/flevoland: 1666 patches

=== Step 3: IoT indexes ===
  IoT/amsterdam: 2 readings
  IoT/rotterdam: 1 readings
  IoT/flevoland: 11 readings

=== Step 4: Download ALL patches to GDrive ===

sentinel2/amsterdam: 1666 patches


100%|███████████████████| 1666/1666 [18:20<00:00,  1.51it/s]


  ✅ 1666/1666 saved to GDrive

sentinel2/rotterdam: 1666 patches


100%|███████████████████| 1666/1666 [18:28<00:00,  1.50it/s]


  ✅ 1666/1666 saved to GDrive

sentinel2/flevoland: 1666 patches


...

  ✅ 419/419 saved to GDrive

thermal/rotterdam: 1132 patches


100%|███████████████████| 1132/1132 [11:07<00:00,  1.70it/s]


  ✅ 1132/1132 saved to GDrive

thermal/flevoland: 501 patches


100%|█████████████████████| 501/501 [04:56<00:00,  1.69it/s]


  ✅ 501/501 saved to GDrive

=== Step 5: Build training index ===
✅ train_index.json saved to GDrive + /content
✅ val_index.json saved to GDrive + /content

=== SUMMARY ===
Total samples:  4287
Train:          3643
Val:            644
OSM coverage:   1507/3643 patches
IoT coverage:   3643/3643 patches

GDrive size: 12G	/gdrive/MyDrive/omnigeofusion/data

✅ ALL data saved to GDrive — never download again!
   Next sessions: data loads from GDrive in ~2-3 mins


In [ ]:
import shutil, os, json
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

GDRIVE_DIR = Path('/gdrive/MyDrive/omnigeofusion/data')
COLAB_DIR  = Path('/content/omnigeofusion_data')
COLAB_DIR.mkdir(parents=True, exist_ok=True)

WORKERS   = 8   # parallel copy workers
LOG_FILE  = '/content/copy_progress.json'  # tracks what's been copied

# ── Load progress log (resume support) ────────────────────────
if os.path.exists(LOG_FILE):
    with open(LOG_FILE) as f:
        completed = set(json.load(f))
    print(f'⚠️  Resuming — {len(completed)} files already copied')
else:
    completed = set()
    print('🆕 Fresh download — starting from scratch')

# ── Count all files ───────────────────────────────────────────
print('\nScanning GDrive...')
all_files = [f for f in GDRIVE_DIR.rglob('*') if f.is_file()]
print(f'Total files in GDrive: {len(all_files)}')

# Filter out already completed
pending = [
    f for f in all_files
    if str(f.relative_to(GDRIVE_DIR)) not in completed
]
print(f'Already copied: {len(all_files) - len(pending)}')
print(f'Remaining:      {len(pending)}')

if not pending:
    print('✅ All files already copied!')
else:
    # ── Thread-safe progress tracking ─────────────────────────
    lock    = threading.Lock()
    copied  = 0
    skipped = 0
    errors  = 0
    done    = list(completed)  # track newly completed

    def copy_file(src: Path) -> tuple:
        """Copy one file from GDrive to /content."""
        rel = src.relative_to(GDRIVE_DIR)
        dst = COLAB_DIR / rel
        try:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(src), str(dst))
            return 'copied', str(rel)
        except Exception as e:
            return 'error', str(rel)

    # ── Save progress every 100 files ─────────────────────────
    def save_progress():
        with open(LOG_FILE, 'w') as f:
            json.dump(done, f)

    # ── Parallel copy with progress bar ───────────────────────
    print(f'\nCopying with {WORKERS} parallel workers...')
    with tqdm(total=len(pending), ncols=70, unit='file') as pbar:
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futures = {ex.submit(copy_file, f): f for f in pending}

            for i, future in enumerate(as_completed(futures)):
                status, rel = future.result()

                with lock:
                    if status == 'copied':
                        copied += 1
                        done.append(rel)
                    elif status == 'error':
                        errors += 1

                    pbar.set_postfix({
                        'copied': copied,
                        'errors': errors,
                    })
                    pbar.update(1)

                    # Save progress every 100 files
                    if (i + 1) % 100 == 0:
                        save_progress()

    # Final save
    save_progress()

    print(f'\n✅ Complete!')
    print(f'   Copied:  {copied}')
    print(f'   Errors:  {errors}')
    print(f'\nSize:  {os.popen("du -sh /content/omnigeofusion_data").read().strip()}')
    print(f'Files: {os.popen("find /content/omnigeofusion_data -type f | wc -l").read().strip()}')
    print(f'\n✅ Progress saved to {LOG_FILE}')
    print('   If interrupted → re-run this cell to resume')

🆕 Fresh download — starting from scratch

Scanning GDrive...
Total files in GDrive: 12845
Already copied: 0
Remaining:      12845

Copying with 8 parallel workers...


100%|█| 12845/12845 [18:09<00:00, 11.79file/s, copied=12845, errors=0]


✅ Complete!
   Copied:  12845
   Errors:  0

Size:  12G	/content/omnigeofusion_data
Files: 12845

✅ Progress saved to /content/copy_progress.json
   If interrupted → re-run this cell to resume


In [ ]:
import subprocess, os
from google.colab import userdata

AWS_KEY    = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET = userdata.get('AWS_SECRET_ACCESS_KEY')

env = os.environ.copy()
env['AWS_ACCESS_KEY_ID']     = AWS_KEY
env['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
env['AWS_DEFAULT_REGION']    = 'us-east-1'
env['PYTHONPATH']            = '/content/omnigeofusion'

os.chdir('/content/omnigeofusion')

process = subprocess.Popen(
    ['/content/venv/bin/python3', '-m', 'src.data.build_train_index',
     '--data-dir',   '/content/omnigeofusion_data',
     '--gdrive-dir', '/gdrive/MyDrive/omnigeofusion/data'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print(f'\nExited: {process.returncode}')

2026-08-06 17:32:37,694 INFO Loading indexes...
2026-08-06 17:32:37,772 INFO amsterdam: S2=1666 SAR=1194 LiDAR=1584 Thermal=419
2026-08-06 17:32:37,772 INFO rotterdam: S2=1666 SAR=814 LiDAR=289 Thermal=1132
2026-08-06 17:32:37,772 INFO flevoland: S2=1666 SAR=1112 LiDAR=981 Thermal=501
2026-08-06 17:32:37,872 INFO sentinel1/amsterdam: 1141/1504 matched
2026-08-06 17:32:37,884 INFO sentinel1/rotterdam: 814/1666 matched
2026-08-06 17:32:37,900 INFO sentinel1/flevoland: 1007/1117 matched
2026-08-06 17:32:37,922 INFO lidar/amsterdam: 1430/1504 matched
2026-08-06 17:32:37,927 INFO lidar/rotterdam: 289/1666 matched
2026-08-06 17:32:37,941 INFO lidar/flevoland: 981/1117 matched
2026-08-06 17:32:37,948 INFO thermal/amsterdam: 393/1504 matched
2026-08-06 17:32:37,967 INFO thermal/rotterdam: 1132/1666 matched
2026-08-06 17:32:37,974 INFO thermal/flevoland: 430/1117 matched
2026-08-06 17:32:37,984 INFO 
=== Coverage (3643 train samples) ===
2026-08-06 17:32:37,985 INFO   S2      : 3643/3643 (100%)

In [ ]:
import subprocess, os
from google.colab import userdata

VENV_PYTHON = '/content/venv/bin/python3'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')
CKPT_DIR    = '/gdrive/MyDrive/omnigeofusion/checkpoints/ssl'
latest      = f'{CKPT_DIR}/ssl_latest.pt'

cmd = [
    VENV_PYTHON, '-m', 'src.training.ssl_pretrain',
    '--model-config', 'configs/model_config.yaml',
    '--train-config', 'configs/training_config.yaml',
    '--data-dir',     '/content/omnigeofusion_data',
]

if os.path.exists(latest):
    print(f'⚠️  Resuming from {latest}')
    cmd += ['--resume', latest]
else:
    print('🆕 Starting fresh SSL pre-training...')

env = os.environ.copy()
env['MPLBACKEND']            = 'agg'
env['AWS_ACCESS_KEY_ID']     = AWS_KEY
env['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
env['AWS_DEFAULT_REGION']    = 'us-east-1'
env['MLFLOW_TRACKING_URI']   = 'http://54.83.1.56:5000'
env['PYTHONPATH']            = '/content/omnigeofusion'

os.chdir('/content/omnigeofusion')

process = subprocess.Popen(
    cmd, env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print(f'\nExited: {process.returncode}')

Finetuning Part


In [ ]:
# Upload SSL best checkpoint to S3
import boto3, os
from google.colab import userdata

AWS_KEY    = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET = userdata.get('AWS_SECRET_ACCESS_KEY')

s3 = boto3.client('s3',
    region_name='us-east-1',
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET
)

BUCKET = 'omnigeofusion-data-288528696055'
best   = '/gdrive/MyDrive/omnigeofusion/checkpoints/ssl/ssl_best.pt'
size   = os.path.getsize(best) / 1e9

print(f'Uploading ssl_best.pt ({size:.1f}GB)...')
s3.upload_file(best, BUCKET, 'models/ssl/ssl_best.pt')
print(f'✅ Uploaded: s3://{BUCKET}/models/ssl/ssl_best.pt')

In [ ]:
# ── Download target indexes from S3 to GDrive + /content ──
import boto3, os
from pathlib import Path
from google.colab import userdata

AWS_KEY    = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET = userdata.get('AWS_SECRET_ACCESS_KEY')

s3 = boto3.client('s3',
    region_name='us-east-1',
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET
)

BUCKET     = 'omnigeofusion-data-288528696055'
DATA_DIR   = Path('/content/omnigeofusion_data')
GDRIVE_DIR = Path('/gdrive/MyDrive/omnigeofusion/data')
AREAS      = ['amsterdam', 'rotterdam', 'flevoland']

for area in AREAS:
    key  = f'netherlands/targets/index/{area}_targets.json'
    # Save to indexes folder
    for dst in [
        DATA_DIR   / 'indexes' / f'{area}_targets.json',
        GDRIVE_DIR / 'indexes' / f'{area}_targets.json',
    ]:
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            s3.download_file(BUCKET, key, str(dst))
            print(f'✅ Downloaded {area}_targets.json → {dst.parent.name}/')
        else:
            print(f'✅ Already exists: {dst.name}')

In [ ]:
import subprocess, os
from google.colab import userdata

VENV_PYTHON = '/content/venv/bin/python3'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')

env = os.environ.copy()
env['AWS_ACCESS_KEY_ID']     = AWS_KEY
env['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
env['AWS_DEFAULT_REGION']    = 'us-east-1'
env['PYTHONPATH']            = '/content/omnigeofusion'

os.chdir('/content/omnigeofusion')

process = subprocess.Popen(
    [VENV_PYTHON, '-m', 'src.data.build_finetune_index',
     '--data-dir',   '/content/omnigeofusion_data',
     '--gdrive-dir', '/gdrive/MyDrive/omnigeofusion/data'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print(f'\nExited: {process.returncode}')

In [ ]:
# ── Run Fine-tuning — All 3 Tasks ──
import subprocess, os
from google.colab import userdata

VENV_PYTHON = '/content/venv/bin/python3'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')
CKPT_DIR    = '/gdrive/MyDrive/omnigeofusion/checkpoints/finetune'
SSL_CKPT    = '/gdrive/MyDrive/omnigeofusion/checkpoints/ssl/ssl_best.pt'

env = os.environ.copy()
env['MPLBACKEND']            = 'agg'
env['AWS_ACCESS_KEY_ID']     = AWS_KEY
env['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
env['AWS_DEFAULT_REGION']    = 'us-east-1'
env['MLFLOW_TRACKING_URI']   = 'http://54.83.1.56:5000'
env['PYTHONPATH']            = '/content/omnigeofusion'

os.chdir('/content/omnigeofusion')

for task in ['urban_change', 'flood_damage', 'agriculture']:
    print(f'\n{"="*50}')
    print(f'Fine-tuning: {task}')
    print(f'{"="*50}')

    # Check for existing checkpoint to resume
    latest = f'{CKPT_DIR}/{task}/latest_phase2.pt'
    if not os.path.exists(latest):
        latest = f'{CKPT_DIR}/{task}/latest_phase1.pt'

    cmd = [
        VENV_PYTHON, '-m', 'src.training.finetune',
        '--task',          task,
        '--model-config',  'configs/model_config.yaml',
        '--train-config',  'configs/training_config.yaml',
        '--data-dir',      '/content/omnigeofusion_data',
        '--ssl-checkpoint', SSL_CKPT,
    ]

    if os.path.exists(latest):
        print(f'⚠️  Resuming from {latest}')
        cmd += ['--resume', latest]
    else:
        print(f'🆕 Starting fresh...')

    process = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    print(f'\n{task} exited: {process.returncode}')

print('\n✅ All tasks fine-tuned!')